In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from collections import Counter
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

CSV = r'C:\Users\User\Documents\Obsidian Vault\03-TRADE-JOURNAL\Bybit-AllPerp-ClosedPNL-1765915200-1769284799 -Bots.csv'
df = pd.read_csv(CSV, encoding='utf-8-sig')

# Parse datetime — format is "HH:MM YYYY-MM-DD"
df['Datetime'] = pd.to_datetime(df['Trade time'], format='%H:%M %Y-%m-%d')
df = df.sort_values('Datetime').reset_index(drop=True)

# Numeric cleanup
for col in ['Order Quantity', 'Entry Price', 'Exit Price', 'Opening Fee', 'Closing Fee', 'Funding Fee', 'Realized P&L']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Derived columns
df['Notional'] = df['Order Quantity'] * df['Exit Price']
df['Total_Fees'] = df['Opening Fee'] + df['Closing Fee']  # trade fees only
df['Gross_PnL'] = df['Realized P&L'] + df['Total_Fees'] - df['Funding Fee']  # back out fees to get gross
df['Win'] = df['Realized P&L'] > 0
df['Loss'] = df['Realized P&L'] < 0
df['CumPnL'] = df['Realized P&L'].cumsum()
df['CumFees'] = df['Total_Fees'].cumsum()
df['CumFunding'] = df['Funding Fee'].cumsum()

# Direction — if exit > entry = long was profitable (grid long), else short
df['Direction'] = np.where(df['Exit Price'] > df['Entry Price'], 'Long', 'Short')

print(f"Loaded {len(df)} grid bot trades")
print(f"Date range: {df['Datetime'].iloc[0]} -> {df['Datetime'].iloc[-1]}")
print(f"Markets: {df['Market'].unique()}")
print(f"\nSample:")
df[['Market', 'Order Quantity', 'Entry Price', 'Exit Price', 'Total_Fees', 'Funding Fee', 'Realized P&L', 'Datetime']].head(10)

In [ ]:
# ═══════════════════════════════════════════════════════
# CORE PERFORMANCE METRICS
# ═══════════════════════════════════════════════════════

total_trades = len(df)
winners = df[df['Win']]
losers = df[df['Loss']]
breakeven = df[(~df['Win']) & (~df['Loss'])]

win_rate = len(winners) / total_trades * 100
total_pnl = df['Realized P&L'].sum()
total_fees = df['Total_Fees'].sum()
total_funding = df['Funding Fee'].sum()
total_gross = df['Gross_PnL'].sum()

avg_win = winners['Realized P&L'].mean() if len(winners) > 0 else 0
avg_loss = losers['Realized P&L'].mean() if len(losers) > 0 else 0
median_win = winners['Realized P&L'].median() if len(winners) > 0 else 0
median_loss = losers['Realized P&L'].median() if len(losers) > 0 else 0
largest_win = df['Realized P&L'].max()
largest_loss = df['Realized P&L'].min()

gross_profit = winners['Realized P&L'].sum() if len(winners) > 0 else 0
gross_loss = abs(losers['Realized P&L'].sum()) if len(losers) > 0 else 1
profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')
payoff_ratio = abs(avg_win / avg_loss) if avg_loss != 0 else float('inf')
expectancy = total_pnl / total_trades

# Duration
date_range = df['Datetime'].iloc[-1] - df['Datetime'].iloc[0]
days = date_range.days + date_range.seconds / 86400

# Avg notional
avg_notional = df['Notional'].mean()

print("="*66)
print("         GRID BOT TRADE DASHBOARD — BYBIT PERPS")
print("="*66)
print(f"  Period:  {df['Datetime'].iloc[0].strftime('%Y-%m-%d')} to {df['Datetime'].iloc[-1].strftime('%Y-%m-%d')}  ({days:.1f} days)")
print(f"  Markets: {', '.join(df['Market'].unique())}")
print("="*66)
print(f"  Total Trades:         {total_trades:>6}")
print(f"  Winners:              {len(winners):>6}  ({win_rate:.1f}%)")
print(f"  Losers:               {len(losers):>6}  ({100-win_rate:.1f}%)")
print(f"  Breakeven:            {len(breakeven):>6}")
print("-"*66)
print(f"  NET REALIZED P&L:   ${total_pnl:>12.2f}")
print(f"  Gross P&L (pre-fee): ${total_gross:>12.2f}")
print(f"  Total Trade Fees:   -${total_fees:>12.2f}")
print(f"  Total Funding Fees:  ${total_funding:>12.2f}  {'(paid)' if total_funding > 0 else '(received)'}")
print(f"  Fee % of Gross:      {(total_fees / total_gross * 100) if total_gross != 0 else 0:>11.1f}%")
print("-"*66)
print(f"  Profit Factor:       {profit_factor:>12.3f}")
print(f"  Payoff Ratio:        {payoff_ratio:>12.3f}  (avg W / avg L)")
print(f"  Expectancy/Trade:    ${expectancy:>12.2f}")
print(f"  Trades/Day:          {total_trades / days:>12.1f}")
print(f"  P&L/Day:             ${total_pnl / days:>12.2f}")
print("-"*66)
print(f"  Avg Winner:          ${avg_win:>12.2f}   Median: ${median_win:>8.2f}")
print(f"  Avg Loser:           ${avg_loss:>12.2f}   Median: ${median_loss:>8.2f}")
print(f"  Largest Win:         ${largest_win:>12.2f}")
print(f"  Largest Loss:        ${largest_loss:>12.2f}")
print(f"  Avg Notional/Trade:  ${avg_notional:>12.2f}")
print("="*66)

In [ ]:
# ═══════════════════════════════════════════════════════
# CONSECUTIVE WIN/LOSS STREAK ANALYSIS
# ═══════════════════════════════════════════════════════

def compute_streaks(series):
    streaks = []
    current = 0
    for val in series:
        if val:
            current += 1
        else:
            if current > 0:
                streaks.append(current)
            current = 0
    if current > 0:
        streaks.append(current)
    return streaks

loss_streaks = compute_streaks(df['Loss'])
win_streaks = compute_streaks(df['Win'])

# Running streak at each trade
df['streak'] = 0
streak = 0
for i in range(len(df)):
    if df.iloc[i]['Loss']:
        streak = streak - 1 if streak < 0 else -1
    elif df.iloc[i]['Win']:
        streak = streak + 1 if streak > 0 else 1
    else:
        streak = 0
    df.iloc[i, df.columns.get_loc('streak')] = streak

max_consec_losses = max(loss_streaks) if loss_streaks else 0
max_consec_wins = max(win_streaks) if win_streaks else 0

# Worst loss streak details
worst_end = df['streak'].idxmin()
worst_len = abs(int(df.loc[worst_end, 'streak']))
worst_start = worst_end - worst_len + 1
worst_trades = df.iloc[worst_start:worst_end+1]
worst_damage = worst_trades['Realized P&L'].sum()

# Best win streak details
best_end = df['streak'].idxmax()
best_len = int(df.loc[best_end, 'streak'])
best_start = best_end - best_len + 1
best_trades = df.iloc[best_start:best_end+1]
best_gain = best_trades['Realized P&L'].sum()

print("="*60)
print("           CONSECUTIVE STREAK ANALYSIS")
print("="*60)
print(f"  Max Consecutive Wins:    {max_consec_wins:>4}  (${best_gain:>10.2f})")
print(f"  Max Consecutive Losses:  {max_consec_losses:>4}  (${worst_damage:>10.2f})")
print(f"  Avg Win Streak:          {np.mean(win_streaks):>6.1f}")
print(f"  Avg Loss Streak:         {np.mean(loss_streaks) if loss_streaks else 0:>6.1f}")
print("-"*60)
print(f"  WORST STREAK: {worst_len} consecutive losses")
print(f"    Damage:  ${worst_damage:.2f}")
print(f"    From:    {worst_trades['Datetime'].iloc[0]}")
print(f"    To:      {worst_trades['Datetime'].iloc[-1]}")
print(f"    Markets: {worst_trades['Market'].unique()}")
print(f"\n  BEST STREAK: {best_len} consecutive wins")
print(f"    Gain:    ${best_gain:.2f}")
print(f"    From:    {best_trades['Datetime'].iloc[0]}")
print(f"    To:      {best_trades['Datetime'].iloc[-1]}")
print("="*60)

# Distribution
print("\nLoss Streak Distribution:")
if loss_streaks:
    for length in sorted(Counter(loss_streaks).keys()):
        cnt = Counter(loss_streaks)[length]
        bar = '#' * cnt
        print(f"  {length:>2} in a row: {cnt:>3}x  {bar}")
else:
    print("  No loss streaks!")

print("\nWin Streak Distribution:")
for length in sorted(Counter(win_streaks).keys()):
    cnt = Counter(win_streaks)[length]
    bar = '#' * cnt
    print(f"  {length:>2} in a row: {cnt:>3}x  {bar}")

In [ ]:
# ═══════════════════════════════════════════════════════
# DRAWDOWN & RISK METRICS
# ═══════════════════════════════════════════════════════

equity = df['CumPnL']
running_max = equity.cummax()
drawdown = equity - running_max
df['Drawdown'] = drawdown.values

max_dd = drawdown.min()
max_dd_idx = drawdown.idxmin()
max_dd_time = df.loc[max_dd_idx, 'Datetime']

peak_before = running_max.loc[:max_dd_idx].max()

# Recovery check
post_dd = equity.loc[max_dd_idx:]
recovered = post_dd[post_dd >= peak_before]
if len(recovered) > 0:
    rec_idx = recovered.index[0]
    rec_trades = rec_idx - max_dd_idx
    recovery_status = f"Recovered ({rec_trades} trades later)"
else:
    recovery_status = "NOT RECOVERED"

# Risk ratios
if df['Realized P&L'].std() > 0:
    sharpe_like = df['Realized P&L'].mean() / df['Realized P&L'].std()
else:
    sharpe_like = 0

downside = df[df['Realized P&L'] < 0]['Realized P&L']
sortino_like = df['Realized P&L'].mean() / downside.std() if len(downside) > 0 and downside.std() > 0 else 0
recovery_factor = total_pnl / abs(max_dd) if max_dd != 0 else 0

# Volatility of returns
daily_pnl = df.groupby(df['Datetime'].dt.date)['Realized P&L'].sum()
daily_std = daily_pnl.std()
daily_mean = daily_pnl.mean()
daily_sharpe = daily_mean / daily_std if daily_std > 0 else 0

print("="*60)
print("              RISK MANAGEMENT METRICS")
print("="*60)
print(f"  Max Drawdown:        ${max_dd:>12.2f}")
print(f"  Max DD Date:          {max_dd_time}")
print(f"  Peak Before DD:      ${peak_before:>12.2f}")
print(f"  Recovery:             {recovery_status}")
print(f"  Recovery Factor:     {recovery_factor:>12.3f}  (net P&L / max DD)")
print("-"*60)
print(f"  Sharpe (per trade):  {sharpe_like:>12.4f}")
print(f"  Sortino (per trade): {sortino_like:>12.4f}")
print("-"*60)
print(f"  Daily P&L Mean:      ${daily_mean:>12.2f}")
print(f"  Daily P&L Std:       ${daily_std:>12.2f}")
print(f"  Daily Sharpe:        {daily_sharpe:>12.4f}")
print(f"  Worst Day:           ${daily_pnl.min():>12.2f}  ({daily_pnl.idxmin()})")
print(f"  Best Day:            ${daily_pnl.max():>12.2f}  ({daily_pnl.idxmax()})")
print("="*60)

In [ ]:
# ═══════════════════════════════════════════════════════
# PER-MARKET BREAKDOWN
# ═══════════════════════════════════════════════════════

market_stats = df.groupby('Market').agg(
    trades=('Realized P&L', 'count'),
    total_pnl=('Realized P&L', 'sum'),
    avg_pnl=('Realized P&L', 'mean'),
    median_pnl=('Realized P&L', 'median'),
    win_rate=('Win', 'mean'),
    total_fees=('Total_Fees', 'sum'),
    total_funding=('Funding Fee', 'sum'),
    avg_notional=('Notional', 'mean'),
    largest_win=('Realized P&L', 'max'),
    largest_loss=('Realized P&L', 'min')
).round(2)
market_stats['win_rate'] = (market_stats['win_rate'] * 100).round(1)
market_stats['fee_drag'] = market_stats['total_fees'] + market_stats['total_funding']
market_stats = market_stats.sort_values('total_pnl', ascending=False)

print("="*80)
print("                    PER-MARKET BREAKDOWN")
print("="*80)
for mkt, row in market_stats.iterrows():
    print(f"\n  {mkt}")
    print(f"    Trades: {int(row['trades']):>5}  |  WR: {row['win_rate']:>5.1f}%  |  Net P&L: ${row['total_pnl']:>10.2f}")
    print(f"    Avg P&L: ${row['avg_pnl']:>8.2f}  |  Median: ${row['median_pnl']:>8.2f}")
    print(f"    Trade Fees: ${row['total_fees']:>8.2f}  |  Funding: ${row['total_funding']:>8.2f}  |  Total Drag: ${row['fee_drag']:>8.2f}")
    print(f"    Largest Win: ${row['largest_win']:>8.2f}  |  Largest Loss: ${row['largest_loss']:>8.2f}")
    print(f"    Avg Notional: ${row['avg_notional']:>10.2f}")
print("\n" + "="*80)

In [ ]:
# ═══════════════════════════════════════════════════════
# FEE & FUNDING DEEP DIVE
# ═══════════════════════════════════════════════════════

print("="*60)
print("             FEE & FUNDING ANALYSIS")
print("="*60)
print(f"  Total Trade Fees (open+close): ${total_fees:>10.2f}")
print(f"  Total Funding Fees:            ${total_funding:>10.2f}")
print(f"  TOTAL FEE DRAG:                ${total_fees + total_funding:>10.2f}")
print(f"  Net Realized P&L:              ${total_pnl:>10.2f}")
print(f"  Gross P&L (no fees):           ${total_gross:>10.2f}")
print("-"*60)
print(f"  Fees as % of Net P&L:          {((total_fees + total_funding) / total_pnl * 100) if total_pnl != 0 else 0:>9.1f}%")
print(f"  Avg Fee/Trade:                 ${(total_fees + total_funding) / total_trades:>10.4f}")
print(f"  Avg Trading Fee/Trade:         ${total_fees / total_trades:>10.4f}")
print(f"  Avg Funding Fee/Trade:         ${total_funding / total_trades:>10.4f}")

# Effective fee rate
total_notional = df['Notional'].sum()
print(f"\n  Total Notional Traded:         ${total_notional:>12.2f}")
print(f"  Effective Trade Fee Rate:       {total_fees / total_notional * 100:.4f}%")
print(f"  Effective Funding Rate:         {total_funding / total_notional * 100:.4f}%")

# Funding by market
print("\n  Funding by Market:")
funding_by_mkt = df.groupby('Market')['Funding Fee'].sum().sort_values()
for mkt, val in funding_by_mkt.items():
    print(f"    {mkt:>20}: ${val:>10.2f}  {'(paid)' if val > 0 else '(received)'}")
print("="*60)

In [ ]:
# ═══════════════════════════════════════════════════════
# DAILY P&L TABLE
# ═══════════════════════════════════════════════════════

daily = df.groupby(df['Datetime'].dt.date).agg(
    trades=('Realized P&L', 'count'),
    pnl=('Realized P&L', 'sum'),
    fees=('Total_Fees', 'sum'),
    funding=('Funding Fee', 'sum'),
    wins=('Win', 'sum'),
    notional=('Notional', 'sum')
).round(2)
daily['wr'] = (daily['wins'] / daily['trades'] * 100).round(1)
daily['cum_pnl'] = daily['pnl'].cumsum().round(2)

print("="*90)
print("                          DAILY P&L BREAKDOWN")
print("="*90)
print(f"{'Date':>12}  {'Trades':>6}  {'WR%':>5}  {'P&L':>10}  {'Fees':>8}  {'Funding':>8}  {'Notional':>12}  {'Cum P&L':>10}")
print("-"*90)
for date, row in daily.iterrows():
    marker = '+' if row['pnl'] > 0 else '-' if row['pnl'] < 0 else ' '
    print(f"  {date}  {int(row['trades']):>5}  {row['wr']:>5.1f}  {marker}${abs(row['pnl']):>9.2f}  ${row['fees']:>7.2f}  ${row['funding']:>7.2f}  ${row['notional']:>11.2f}  ${row['cum_pnl']:>9.2f}")
print("="*90)

# Winning vs losing days
winning_days = (daily['pnl'] > 0).sum()
losing_days = (daily['pnl'] < 0).sum()
print(f"\n  Winning Days: {winning_days}/{len(daily)} ({winning_days/len(daily)*100:.0f}%)")
print(f"  Losing Days:  {losing_days}/{len(daily)} ({losing_days/len(daily)*100:.0f}%)")
print(f"  Avg Winning Day: ${daily[daily['pnl']>0]['pnl'].mean():.2f}")
print(f"  Avg Losing Day:  ${daily[daily['pnl']<0]['pnl'].mean():.2f}")

In [ ]:
# ═══════════════════════════════════════════════════════
# GRID BOT SPECIFIC — PRICE SPACING & FILL ANALYSIS
# ═══════════════════════════════════════════════════════

# Price move per trade (grid capture)
df['Price_Move_%'] = ((df['Exit Price'] - df['Entry Price']) / df['Entry Price'] * 100)

# Per-market grid analysis
print("="*70)
print("         GRID SPACING & CAPTURE ANALYSIS")
print("="*70)
for mkt in df['Market'].unique():
    mkt_df = df[df['Market'] == mkt]
    moves = mkt_df['Price_Move_%']
    pnls = mkt_df['Realized P&L']
    print(f"\n  {mkt} ({len(mkt_df)} trades)")
    print(f"    Avg Price Move:   {moves.mean():>+7.3f}%  (grid capture per fill)")
    print(f"    Median Move:      {moves.median():>+7.3f}%")
    print(f"    Move Range:       {moves.min():>+7.3f}% to {moves.max():>+7.3f}%")
    print(f"    Avg P&L/fill:     ${pnls.mean():>8.2f}")
    
    # How often does it close in profit vs at a loss
    w = (mkt_df['Win']).sum()
    l = (mkt_df['Loss']).sum()
    print(f"    Win/Loss:         {w}W / {l}L ({w/(w+l)*100:.0f}% WR)" if (w+l) > 0 else "")
    
    # Time between fills
    if len(mkt_df) > 1:
        time_diffs = mkt_df['Datetime'].diff().dropna()
        avg_mins = time_diffs.mean().total_seconds() / 60
        print(f"    Avg Time/Fill:    {avg_mins:.0f} min ({avg_mins/60:.1f} hrs)")

print("\n" + "="*70)

In [ ]:
# ═══════════════════════════════════════════════════════
# VISUAL DASHBOARD
# ═══════════════════════════════════════════════════════

fig = plt.figure(figsize=(18, 22), facecolor='#1a1a2e')
gs = GridSpec(5, 2, figure=fig, hspace=0.4, wspace=0.3)

def style_ax(ax, title):
    ax.set_facecolor('#16213e')
    ax.set_title(title, color='white', fontsize=12, fontweight='bold', pad=10)
    ax.tick_params(colors='#aaa')
    for spine in ax.spines.values(): spine.set_color('#333')
    ax.grid(True, alpha=0.15, color='white')

# 1. Equity Curve
ax1 = fig.add_subplot(gs[0, :])
ax1.fill_between(range(len(equity)), equity, alpha=0.3, color='#00d4aa')
ax1.plot(equity.values, color='#00d4aa', linewidth=1.5)
ax1.axhline(y=0, color='#666', linestyle='--', linewidth=0.8)
ax1.scatter([max_dd_idx], [equity.iloc[max_dd_idx]], color='#ff4757', s=100, zorder=5, 
            label=f'Max DD: ${max_dd:.0f}')
style_ax(ax1, 'EQUITY CURVE — Grid Bot Realized P&L')
ax1.set_ylabel('Cumulative P&L ($)', color='#aaa')
ax1.legend(facecolor='#16213e', edgecolor='#333', labelcolor='white')

# 2. Drawdown
ax2 = fig.add_subplot(gs[1, :])
ax2.fill_between(range(len(drawdown)), drawdown, alpha=0.5, color='#ff4757')
ax2.plot(drawdown.values, color='#ff6b81', linewidth=0.8)
style_ax(ax2, 'DRAWDOWN')
ax2.set_ylabel('Drawdown ($)', color='#aaa')

# 3. P&L per trade (bar)
ax3 = fig.add_subplot(gs[2, 0])
colors_bar = ['#00d4aa' if x > 0 else '#ff4757' for x in df['Realized P&L']]
ax3.bar(range(len(df)), df['Realized P&L'], color=colors_bar, width=1.0, alpha=0.7)
style_ax(ax3, 'P&L PER TRADE')
ax3.set_xlabel('Trade #', color='#aaa')
ax3.set_ylabel('P&L ($)', color='#aaa')

# 4. P&L Distribution
ax4 = fig.add_subplot(gs[2, 1])
ax4.hist(df['Realized P&L'], bins=50, color='#5dade2', alpha=0.7, edgecolor='#333')
ax4.axvline(x=0, color='white', linestyle='--', linewidth=0.8)
ax4.axvline(x=df['Realized P&L'].mean(), color='#ffd32a', linewidth=1.5, label=f'Mean: ${expectancy:.2f}')
ax4.axvline(x=df['Realized P&L'].median(), color='#ff4757', linewidth=1.5, label=f'Median: ${df["Realized P&L"].median():.2f}')
style_ax(ax4, 'P&L DISTRIBUTION')
ax4.legend(facecolor='#16213e', edgecolor='#333', labelcolor='white', fontsize=9)

# 5. Cumulative by Market
ax5 = fig.add_subplot(gs[3, 0])
market_colors = {'RIVERUSDT': '#00d4aa', 'AXSUSDT': '#5dade2', 'RENDERUSDT': '#ffd32a', 
                 'PIPPINUSDT': '#ff4757', 'JELLYJELLYUSDT': '#c39bd3'}
for mkt in df['Market'].unique():
    mkt_df = df[df['Market'] == mkt]
    cum = mkt_df['Realized P&L'].cumsum()
    ax5.plot(range(len(cum)), cum.values, color=market_colors.get(mkt, '#aaa'), 
             linewidth=1.5, label=f'{mkt}: ${mkt_df["Realized P&L"].sum():.0f}')
style_ax(ax5, 'CUMULATIVE P&L BY MARKET')
ax5.legend(facecolor='#16213e', edgecolor='#333', labelcolor='white', fontsize=8)
ax5.set_ylabel('Cum P&L ($)', color='#aaa')

# 6. Market share (pie)
ax6 = fig.add_subplot(gs[3, 1])
mkt_pnl = df.groupby('Market')['Realized P&L'].sum().sort_values(ascending=False)
mkt_counts = df['Market'].value_counts()
pie_colors = [market_colors.get(m, '#aaa') for m in mkt_counts.index]
ax6.pie(mkt_counts.values, labels=mkt_counts.index, autopct='%1.1f%%', 
        colors=pie_colors, textprops={'color': 'white', 'fontsize': 9})
style_ax(ax6, 'TRADE COUNT BY MARKET')
ax6.grid(False)

# 7. Daily P&L bars
ax7 = fig.add_subplot(gs[4, 0])
daily_colors = ['#00d4aa' if x > 0 else '#ff4757' for x in daily['pnl']]
ax7.bar(range(len(daily)), daily['pnl'], color=daily_colors, alpha=0.8)
ax7.set_xticks(range(len(daily)))
ax7.set_xticklabels([str(d) for d in daily.index], rotation=45, ha='right', fontsize=7)
style_ax(ax7, 'DAILY P&L')
ax7.set_ylabel('P&L ($)', color='#aaa')

# 8. Fee breakdown stacked
ax8 = fig.add_subplot(gs[4, 1])
fee_by_mkt = df.groupby('Market').agg(trade_fees=('Total_Fees', 'sum'), funding=('Funding Fee', 'sum'))
fee_by_mkt = fee_by_mkt.sort_values('trade_fees', ascending=True)
x_pos = range(len(fee_by_mkt))
ax8.barh(list(x_pos), fee_by_mkt['trade_fees'], color='#ff4757', alpha=0.8, label='Trade Fees')
ax8.barh(list(x_pos), fee_by_mkt['funding'], left=fee_by_mkt['trade_fees'], color='#ffd32a', alpha=0.8, label='Funding')
ax8.set_yticks(list(x_pos))
ax8.set_yticklabels(fee_by_mkt.index)
style_ax(ax8, 'FEE BREAKDOWN BY MARKET')
ax8.set_xlabel('Fees ($)', color='#aaa')
ax8.legend(facecolor='#16213e', edgecolor='#333', labelcolor='white', fontsize=9)

fig.suptitle('GRID BOT DASHBOARD — BYBIT PERPS', color='white', fontsize=16, fontweight='bold', y=0.99)
plt.savefig(r'C:\Users\User\Documents\Obsidian Vault\03-TRADE-JOURNAL\grid_bot_dashboard.png', 
            dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print("Dashboard saved.")

In [ ]:
# ═══════════════════════════════════════════════════════
# TOP / BOTTOM TRADES & EXECUTIVE SUMMARY
# ═══════════════════════════════════════════════════════

print("="*80)
print("  TOP 10 BEST TRADES")
print("="*80)
best = df.nlargest(10, 'Realized P&L')[['Market', 'Order Quantity', 'Entry Price', 'Exit Price', 'Realized P&L', 'Funding Fee', 'Datetime']].copy()
best['Realized P&L'] = best['Realized P&L'].apply(lambda x: f'${x:.2f}')
print(best.to_string(index=False))

print(f"\n{'='*80}")
print("  TOP 10 WORST TRADES")
print("="*80)
worst = df.nsmallest(10, 'Realized P&L')[['Market', 'Order Quantity', 'Entry Price', 'Exit Price', 'Realized P&L', 'Funding Fee', 'Datetime']].copy()
worst['Realized P&L'] = worst['Realized P&L'].apply(lambda x: f'${x:.2f}')
print(worst.to_string(index=False))

print(f"\n\n{'='*66}")
print("                  EXECUTIVE SUMMARY")
print("="*66)
print(f"  {total_trades} grid bot fills over {days:.1f} days")
print(f"  Net Realized P&L:    ${total_pnl:>10.2f}")
print(f"  Total Fee Drag:      ${total_fees + total_funding:>10.2f}")
print(f"  Win Rate:            {win_rate:.1f}%")
print(f"  Profit Factor:       {profit_factor:.3f}")
print(f"  Expectancy:          ${expectancy:.2f}/trade")
print(f"  P&L/Day:             ${total_pnl/days:.2f}")
print(f"  Max Drawdown:        ${max_dd:.2f}")
print(f"  Max Consec. Losses:  {max_consec_losses}")
print(f"  Recovery Factor:     {recovery_factor:.3f}")
print(f"\n  VERDICT: {'PROFITABLE' if total_pnl > 0 else 'UNPROFITABLE'}")
print("="*66)